# Étape 99 (2/3) — export du classeur Excel final

Dernier maillon : relit toutes les tables déjà calculées (étapes 04-06, `indicateurs.py`,
`intervalle_confiance.py`) et les met en forme dans UN classeur Excel, `bkt_final.xlsx` - 6 feuilles
(voir `export_excel.py`), construites avec l'utilitaire `xlsx.py` (formats, tableaux, graphiques).

## 0. Configuration — [`pipeline.py`](pipeline.py)

In [1]:
import sys
from pathlib import Path

ICI = Path.cwd()
sys.path.insert(0, str(ICI))
RACINE = ICI.parents[1]

import pandas as pd
pd.set_option("display.width", 160)

DOSSIER_BKT = RACINE / "data" / "donnees_valides" / "bkt"

## 1. Résultat déjà préparé (ou à préparer) — [`pipeline.py`](pipeline.py)

In [2]:
if not (DOSSIER_BKT / "bkt_final.xlsx").exists():
    print("Pas encore construit - lancement de pipeline.py.")
    import runpy
    runpy.run_path(str(ICI / "pipeline.py"), run_name="__main__")
else:
    print(f"déjà construit : {DOSSIER_BKT / 'bkt_final.xlsx'}")

déjà construit : C:\Users\olivi\Documents\GitHub\LEVEL-Cyclist-Risk-Estimation\VV6\data\donnees_valides\bkt\bkt_final.xlsx


## 2. Les 6 feuilles — [`export_excel.py`](export_excel.py) / [`xlsx.py`](xlsx.py)

    Résumé                   BKT observé/extrapolé national, % extrapolé, IC 95%
    Indicateurs de base      toutes les statistiques (valeurs brutes + indice base 100), 3 graphiques
    Détail par cluster       chaque métrique décomposée par cluster K4 + colonne Total national
    Intervalle de confiance  détail du bootstrap, méthode standard
    IC BKT mini              même bootstrap, panel équilibré
    Test de croissance       bootstrap apparié, significativité de la croissance

In [3]:
import openpyxl

classeur = openpyxl.load_workbook(DOSSIER_BKT / "bkt_final.xlsx")
print("Feuilles :", classeur.sheetnames)

ws = classeur["Résumé"]
for ligne in ws.iter_rows(min_row=1, max_row=11, values_only=True):
    valeurs = [v for v in ligne if v is not None]
    if valeurs:
        print(valeurs)

Feuilles : ['Résumé', 'Indicateurs de base', 'Détail par cluster', 'Intervalle de confiance', 'IC BKT mini', 'Test de croissance']
['BKT final : avec n, Z complet, rognage LOOCV par cluster']
['Quelle confiance accorder au résultat ? BKT observé/extrapolé national avec IC 95% (incertitude de NOTRE méthode, voir le README), et test de significativité de la croissance. 3000 tirages bootstrap par année/comparaison.']
['Année', 'BKT observé', 'BKT_ext (point)', '% extrapolé', 'Médiane bootstrap', 'IC bas (2.5%)', 'IC haut (97.5%)', 'Largeur relative (%)']
[2019, 2.385946319682768, 6.143230950290459, 61.16137682289223, 6.178380717922481, 5.208298603557914, 7.391814331570854, 35.34122980927535]
[2020, 3.385851577923911, 7.475248732374852, 54.70583389071735, 7.573981689278407, 7.023413906589061, 8.663725449616447, 21.65718918160816]
[2021, 4.944309913307186, 9.717233848988649, 49.11813392427743, 9.7940526139999, 9.180969618837898, 10.71345398083989, 15.64709137677508]
[2022, 5.400607107257074

## 3. Vérification de cohérence : le classeur reproduit-il les CSV sources ?

Contrôle interne : la feuille "Résumé" du classeur ne fait que RELIRE `indicateurs.csv`/`ic_standard.csv`
et les mettre en forme - ses valeurs doivent être EXACTEMENT celles des CSV, pas une resaisie.

In [4]:
def lignes_donnees(feuille):
    return [ligne for ligne in feuille.iter_rows(min_row=1, max_row=15, values_only=True) if ligne[0] and isinstance(ligne[0], (int, float)) and ligne[0] > 2000]

indicateurs = pd.read_csv(DOSSIER_BKT / "indicateurs.csv")
valeurs_classeur = {int(ligne[0]): round(ligne[2], 3) for ligne in lignes_donnees(ws)}
valeurs_csv = dict(zip(indicateurs["annee"], indicateurs["bkt_ext_Mdkm"].round(3)))
print("classeur == CSV pour chaque année :", valeurs_classeur == valeurs_csv)
assert valeurs_classeur == valeurs_csv, "le classeur Excel diverge du CSV source - incohérence"

classeur == CSV pour chaque année : True


## Bilan

Le classeur Excel reproduit EXACTEMENT les valeurs des tables CSV déjà vérifiées (étapes 04-06,
`indicateurs.py`, `intervalle_confiance.py`) - c'est une mise en forme, pas un recalcul.
`export_excel.py`/`xlsx.py` sont deux modules RÉUTILISABLES (classe `Classeur`, fonctions
`serie`/`plage`/`tableau`/`graphique`) : toute nouvelle table du pipeline peut être ajoutée au classeur
en quelques lignes, sans réinventer la mise en forme.